In [0]:
--- Data checking and familiarizing with data
SELECT * FROM workspace.bright_tv.user_profiles;
SELECT * FROM  workspace.bright_tv.viewership;

--------------------------------------------------
        --DATA CLEANSING AND TRANSFORMATION
--------------------------------------------------

--Drop less informative colums fro profile table
ALTER TABLE workspace.bright_tv.user_profiles
DROP COLUMN Name,Surname,Email,`Social Media Handle`;

--Count how many rows are complete, fully empty and patially empty

SELECT row_quality, COUNT(*) AS count_rows
FROM (
    SELECT *,
           CASE 
               WHEN  Gender = 'None' AND Race = 'None' AND Age = 0
                    AND Province = 'None'
               THEN 'Fully Empty'
               WHEN Gender = 'None' OR Race = 'None' OR Age IS NULL OR Province = 'None'           
                THEN 'Partial Missing'
               ELSE 'Complete'
           END AS row_quality
    FROM workspace.bright_tv.user_profiles
) 
GROUP BY row_quality;
-- Clean the tables
ALTER TABLE workspace.bright_tv.viewership
DROP COLUMN userid4;

  

------------------------------------------------------
                   ----Processing
------------------------------------------------------
-- Join Profile and Viewership database tables and name it Bright_TV_table 


CREATE TABLE Bright_TV_table AS
FROM workspace.bright_tv.viewership AS A
JOIN workspace.bright_tv.user_profiles AS B
ON A.UserID0 = B.UserID;

-- data processing 
REPLACE TABLE Bright_TV_table AS
SELECT*,
date_format (Duration2, 'HH:mm:ss') AS Viewership_Duration
FROM  Bright_TV_table;
SELECT 
    COALESCE(NULLIF(Gender, 'None'), 'Unknown') AS Gender,
    COALESCE(NULLIF(Race, 'None'), 'Unknown') AS Race,
    COALESCE(NULLIF(Province, 'None'), 'Unknown') AS Province
FROM Bright_TV_table;

--Total Viewership per channel
SELECT 
    Channel2,
    SUM(
        CAST(SUBSTRING(Viewership_Duration,1,2) AS INT) * 3600 +
        CAST(SUBSTRING(Viewership_Duration,4,2) AS INT) * 60 +
        CAST(SUBSTRING(Viewership_Duration,7,2) AS INT)
    ) AS Total_streaming_duration
FROM Bright_TV_table
GROUP BY Channel2
ORDER BY Total_streaming_duration DESC;

-- Average View Time per Channel
SELECT 
    Channel2,
    AVG(
        CAST(SUBSTRING(Viewership_Duration, 1,2) AS INT) * 3600 +
        CAST(SUBSTRING(Viewership_Duration, 4,2) AS INT) * 60 +
        CAST(SUBSTRING(Viewership_Duration, 7,2) AS INT)
    ) AS Avg_Streaming_time 
FROM Bright_TV_table
GROUP BY Channel2
ORDER BY Avg_Streaming_time DESC;

--Viewership by Gender
SELECT 
    Gender,
    COUNT(*) AS Streaming_By_Gender
FROM Bright_TV_table
GROUP BY Gender
ORDER BY Streaming_By_Gender DESC;

---testing
SELECT 
    Gender,

    -- Number of streams
    COUNT(*) AS `Streaming By Gender`,

    -- Total streaming duration (in seconds)
    SUM(
        CAST(SUBSTRING(Viewership_Duration,1,2) AS INT) * 3600 +
        CAST(SUBSTRING(Viewership_Duration,4,2) AS INT) * 60 +
        CAST(SUBSTRING(Viewership_Duration,7,2) AS INT)
    ) AS Total_Streaming_Duration

FROM Bright_TV_table
GROUP BY Gender
ORDER BY `Streaming By Gender` DESC;

-- Streaming Viewership by Province
SELECT 
    Province,
    COUNT(*) AS Total_Streaming_By_Province
FROM Bright_TV_table
WHERE Province IS NOT NULL
GROUP BY Province
ORDER BY Total_Streaming_By_Province DESC;

-- Most stream channel by Province
SELECT 
    Province,
    Channel2,
    COUNT(*) AS `Most stream channel by Province`
FROM Bright_TV_table
WHERE Province IS NOT NULL
GROUP BY Province, Channel2
ORDER BY Province, `Most stream channel by Province` DESC;

-- Streaming by Time of Day (convert time to SA Time)
--------------------------------------------------------------------
--             Streaming by Time of Day

--------------------------------------------------------------------
 -------convert time to SA Time-----------------

WITH base AS (
    SELECT *,
           date_format(
               from_utc_timestamp(
                   to_timestamp(RecordDate2, 'yyyy/MM/dd HH:mm'),
                   'Africa/Johannesburg'
               ),
               'HH:mm:ss'
           ) AS Streaming_SA_Time
    FROM Bright_TV_table
)
---------------------------Streaming by Time of Day-----------------------------------------
SELECT
CASE
    WHEN CAST(SUBSTRING(Streaming_SA_Time, 1, 2) AS INT) BETWEEN 0 AND 5 THEN 'Early Morning'
    WHEN CAST(SUBSTRING(Streaming_SA_Time, 1, 2) AS INT) BETWEEN 6 AND 8 THEN 'Morning Rush'
    WHEN CAST(SUBSTRING(Streaming_SA_Time, 1, 2) AS INT) BETWEEN 9 AND 11 THEN 'Mid Morning'
    WHEN CAST(SUBSTRING(Streaming_SA_Time, 1, 2) AS INT) BETWEEN 12 AND 14 THEN 'Mid Day'
    WHEN CAST(SUBSTRING(Streaming_SA_Time, 1, 2) AS INT) BETWEEN 15 AND 16 THEN 'Afternoon'
    WHEN CAST(SUBSTRING(Streaming_SA_Time, 1, 2) AS INT) BETWEEN 17 AND 19 THEN 'Evening'
    WHEN CAST(SUBSTRING(Streaming_SA_Time, 1, 2) AS INT) BETWEEN 19 AND 21 THEN 'Prime Time'
    ELSE 'Night'
END AS time_bucket,
COUNT(*) AS `Streamings per time bucket`
FROM base
GROUP BY time_bucket;

-----------------Most streamed channel by Age Bucket----------
----combined

WITH base AS (
    SELECT *,
    CASE 
        WHEN Age BETWEEN 0 AND 12 THEN 'Kids'
        WHEN Age BETWEEN 13 AND 19 THEN 'Teens'
        WHEN Age BETWEEN 20 AND 40 THEN 'Young Adults'
        WHEN Age > 40 THEN 'Adults'
        ELSE 'Unknown'
    END AS `Age Group Bucket`
    FROM Bright_TV_table
)

-- Most streamed channels by Age Group
SELECT
    `Age Group Bucket`,
    Channel2,
    COUNT(*) AS `Total_streams per age group`
FROM base
WHERE `Age Group Bucket` IS NOT NULL
GROUP BY `Age Group Bucket`, Channel2
ORDER BY `Age Group Bucket`, `Total_streams per age group` DESC;

select *
from Bright_TV_table;












